# Principle 2: Triadic Closure

## Experiment Configuration

Set models, simulation sizes, temperatures, environments, and CoT options here before running the experiment cells.


### Configuration Cell

このセルで Principle 2 の実験条件を一括管理します。`EXPERIMENTS` の各要素が1つの実験条件で、`summary_group` によって通常の SBM 初期化、ER 初期化、CoT 条件を分けて解析します。


In [ ]:
# Shared settings.
BASELINE_MODEL = 'Qwen/Qwen3.5-4B'
MODEL_NAMES = [
    'gpt-5-nano',
    'Qwen/Qwen3.5-4B',
    'Qwen/Qwen3.5-0.8B',
]
DEFAULT_TEMPERATURES = [None]
DEFAULT_COT_CONFIG = {'max_new_tokens': 16384, 'qwen_enable_thinking': True}
COT_RETRY_MAX_NEW_TOKENS = 32768
ENVIRONMENTS = [
    ('school', 'classmates'),
    ('work', 'colleagues'),
    ('community', 'neighbors'),
]

DEFAULT_PARAMETERS = dict(n_min=50, n_max=50, n_step=1, num_simulations=1)

# Each dictionary is one runnable experiment condition.
# environment=None means the baseline prompt without a school/work/community context.
EXPERIMENTS = [
    *[
        {
            'name': f'sbm_model_{model.replace("/", "-")}_baseline',
            'summary_group': 'sbm',
            'model': model,
            'environment': None,
            'num_common_neighbors': False,
            'er': False,
            'COT': False,
            'temperatures': DEFAULT_TEMPERATURES,
            'parameters': DEFAULT_PARAMETERS,
            'analyze_detail': model == BASELINE_MODEL,
        }
        for model in MODEL_NAMES
    ],
    *[
        {
            'name': f'sbm_model_{BASELINE_MODEL.replace("/", "-")}_{environment[0]}_{environment[1]}',
            'summary_group': 'sbm',
            'model': BASELINE_MODEL,
            'environment': environment,
            'num_common_neighbors': False,
            'er': False,
            'COT': False,
            'temperatures': DEFAULT_TEMPERATURES,
            'parameters': DEFAULT_PARAMETERS,
        }
        for environment in ENVIRONMENTS
    ],
    {
        'name': f'sbm_model_{BASELINE_MODEL.replace("/", "-")}_baseline_common_neighbor_count',
        'summary_group': 'sbm',
        'model': BASELINE_MODEL,
        'environment': None,
        'num_common_neighbors': True,
        'er': False,
        'COT': False,
        'temperatures': DEFAULT_TEMPERATURES,
        'parameters': DEFAULT_PARAMETERS,
        'include_in_summary': False,
    },
    {
        'name': f'er_model_{BASELINE_MODEL.replace("/", "-")}_baseline',
        'summary_group': 'er',
        'model': BASELINE_MODEL,
        'environment': None,
        'num_common_neighbors': False,
        'er': True,
        'COT': False,
        'temperatures': DEFAULT_TEMPERATURES,
        'parameters': DEFAULT_PARAMETERS,
        'analyze_detail': True,
    },
    {
        'name': f'sbm_model_{BASELINE_MODEL.replace("/", "-")}_baseline_cot',
        'summary_group': 'cot',
        'model': BASELINE_MODEL,
        'environment': None,
        'num_common_neighbors': False,
        'er': False,
        'COT': True,
        'cot_config': DEFAULT_COT_CONFIG,
        'temperatures': DEFAULT_TEMPERATURES,
        'parameters': DEFAULT_PARAMETERS,
    },
]

RUN_EXPERIMENTS = True
RUN_ANALYSIS = True
IGNORE_EXISTING_OUTPUTS = False

print("Configuration done.")


Configuration done.


### Colab Repository Setup

This cell clones or updates the repository in local Colab storage, mounts Google Drive, creates a persistent output directory in Drive, and installs `requirements.txt`. Keeping the working directory under `/content` avoids Google Drive mount disconnect errors during imports while results are saved under `/content/drive/MyDrive/llm-network-formation-outputs`.

In [2]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

REPO_DIR = Path('/content/llm-network-formation')
OUTPUT_DIR = Path('/content/drive/MyDrive/llm-network-formation-outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if 'DEFAULT_PROFILES_FILENAME' in globals():
    DEFAULT_PROFILES_FILENAME = str(OUTPUT_DIR / 'profiles.jsonl')
if 'LUCKY_NUMBER_PROFILES_FILENAME' in globals():
    LUCKY_NUMBER_PROFILES_FILENAME = str(OUTPUT_DIR / 'profiles_with_lucky_number.jsonl')
if 'EXPERIMENTS' in globals():
    for experiment in EXPERIMENTS:
        profile_path = experiment.get('profiles_filename')
        if profile_path is not None and Path(profile_path).name == 'profiles.jsonl':
            experiment['profiles_filename'] = str(OUTPUT_DIR / 'profiles.jsonl')
        if profile_path is not None and Path(profile_path).name == 'profiles_with_lucky_number.jsonl':
            experiment['profiles_filename'] = str(OUTPUT_DIR / 'profiles_with_lucky_number.jsonl')



if REPO_DIR.exists():
    %cd $REPO_DIR
    !git pull --ff-only
else:
    %cd /content
    !git clone https://github.com/yohei-kobashi/llm-network-formation.git
    %cd $REPO_DIR

!pip uninstall -y vllm transformers torch torchvision torchaudio xformers triton cuda-toolkit cuda-bindings cuda-python nvidia-cublas nvidia-cuda-cupti nvidia-cuda-nvrtc nvidia-cuda-runtime nvidia-cudnn-cu13 nvidia-cudnn-cu12 nvidia-cufft nvidia-cufile nvidia-curand nvidia-cusolver nvidia-cusparse nvidia-cusparselt-cu13 nvidia-cusparselt-cu12 nvidia-nccl-cu13 nvidia-nccl-cu12 nvidia-nvjitlink nvidia-nvshmem-cu13 nvidia-nvshmem-cu12 nvidia-nvtx
# Avoid upgrading setuptools in an already-running Colab runtime; it can trigger a restart warning for _distutils_hack.
!pip install -U pip wheel uv
# Colab already provides numpy/scipy; upgrading them after import triggers runtime restart warnings.
!pip install matplotlib networkx pandas seaborn netgraph powerlaw openai anthropic replicate
VLLM_VERSION = '0.20.0'
VLLM_CUDA_VERSION = '129'
VLLM_WHEEL = f'https://github.com/vllm-project/vllm/releases/download/v{VLLM_VERSION}/vllm-{VLLM_VERSION}%2Bcu{VLLM_CUDA_VERSION}-cp38-abi3-manylinux_2_31_x86_64.whl'
print('Installing vLLM:', VLLM_VERSION, 'CUDA wheel:', f'cu{VLLM_CUDA_VERSION}')
# vLLM 0.20.0 switched the default PyPI wheel to CUDA 13.0. Colab currently needs the explicit CUDA 12.9 wheel.
!uv pip install --system --index-strategy unsafe-best-match {VLLM_WHEEL} --extra-index-url=https://download.pytorch.org/whl/cu{VLLM_CUDA_VERSION}
!uv pip install --system --torch-backend=cu{VLLM_CUDA_VERSION} --index-strategy unsafe-best-match 'transformers>=5,<6' accelerate sentencepiece
import importlib
import numpy
import scipy
import netgraph
import torch
try:
    import transformers
except ModuleNotFoundError:
    import sys, subprocess
    print('transformers is missing after setup; installing fallback dependencies with pip.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers>=5,<6', 'accelerate', 'sentencepiece'])
    import transformers
print('dependency check ok', numpy.__version__, scipy.__version__, transformers.__version__)
print('torch check', torch.__version__, torch.version.cuda)
try:
    vllm = importlib.import_module('vllm')
    from vllm import LLM, SamplingParams
    from vllm.sampling_params import StructuredOutputsParams
    print('vllm kernel import ok', vllm.__version__)
except Exception as e:
    print(
        'WARNING: vLLM installation completed, but this kernel cannot import CUDA-backed vLLM symbols. '
        'The experiments will continue with the Transformers fallback for Qwen models. '
        f'Kernel import error: {e}'
    )


Mounted at /content/drive
/content
Cloning into 'llm-network-formation'...
remote: Enumerating objects: 480, done.
remote: Counting objects: 100% (373/373), done.
remote: Compressing objects: 100% (225/225), done.
remote: Total 480 (delta 232), reused 285 (delta 145), pack-reused 107 (from 1)
Receiving objects: 100% (480/480), 144.05 MiB | 24.66 MiB/s, done.
Resolving deltas: 100% (272/272), done.
/content/llm-network-formation
Found existing installation: transformers 5.12.0
Uninstalling transformers-5.12.0:
  Successfully uninstalled transformers-5.12.0
Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installat

### API Credentials

This cell reads secrets from Colab userdata and sets environment variables for OpenAI and Hugging Face access. `OPENAI_API_KEY` is required for GPT models; `HF_TOKEN` is optional but useful for Hugging Face model downloads.

In [ ]:
from google.colab import userdata
import os

try:
    openai_api_key = userdata.get('OPENAI_API_KEY')
    hf_token = userdata.get('HF_TOKEN')
except Exception as e:
    print(f'Could not load Colab userdata secrets: {e}')
    openai_api_key = os.environ.get('OPENAI_API_KEY')
    hf_token = os.environ.get('HF_TOKEN')

if openai_api_key:
    os.environ['OPENAI_API_KEY'] = openai_api_key
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
if not os.environ.get('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY is required. Set it in Colab secrets or the environment.')


### Helper Functions and Analysis Code

This cell imports dependencies and defines the Principle 2 network formation, triadic-closure analysis, and table helpers.


In [4]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import json
import random
import os
import collections
import re
import copy
import netgraph
import seaborn as sns
import scipy.stats
from utils import filter_supported_models, get_response, get_responses, summarize_generation_budget_calibration, summarize_reasons

MEDIUM_SIZE = 26
SMALL_SIZE = 0.85 * MEDIUM_SIZE
BIGGER_SIZE = 1.5 * MEDIUM_SIZE

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=0.7*SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=0.7*SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title


def build_response_schema(candidate_names):
    schema_name_enum = list(candidate_names)
    schema_name_type = "integer" if all(isinstance(name, int) for name in candidate_names) else "string"

    return {
        "type": "object",
        "properties": {
            "name": {
                "type": schema_name_type,
                "enum": schema_name_enum,
            },
            "reason": {
                "type": "string",
            },
        },
        "required": ["name", "reason"],
        "additionalProperties": False,
    }


def build_response_list_schema(candidate_names, max_items=4):
    return {
        "type": "array",
        "items": build_response_schema(candidate_names),
        "minItems": 1,
        "maxItems": max_items,
    }


def build_acceptance_response_schema():
    return {
        "type": "object",
        "properties": {
            "accept": {"type": "boolean"},
            "reason": {"type": "string"},
        },
        "required": ["accept", "reason"],
        "additionalProperties": False,
    }


def _clean_model_json_text(text):
    text = text.strip()
    if 'FINAL_JSON:' in text:
        text = text.rsplit('FINAL_JSON:', 1)[1].strip()
    text = re.sub(r"(?is)<think>.*?</think>", "", text).strip()
    return text


def _balanced_json_fragment(text, opener, closer):
    start = text.find(opener)
    if start == -1:
        return None
    depth = 0
    in_string = False
    escape = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch == opener:
            depth += 1
        elif ch == closer:
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None


def first_json_object(text):
    text = _clean_model_json_text(text)
    fenced = re.search(r"(?is)```(?:json)?\s*(\{.*?\})\s*```", text)
    direct_candidates = [text]
    if fenced:
        direct_candidates.insert(0, fenced.group(1).strip())

    for candidate in direct_candidates:
        if not candidate:
            continue
        try:
            return json.loads(candidate)
        except Exception:
            pass

    fragment = _balanced_json_fragment(text, "{", "}")
    if fragment is None:
        return None
    try:
        return json.loads(fragment)
    except Exception:
        return None


def first_json_array(text):
    text = _clean_model_json_text(text)
    fenced = re.search(r"(?is)```(?:json)?\s*(\[.*?\])\s*```", text)
    direct_candidates = [text]
    if fenced:
        direct_candidates.insert(0, fenced.group(1).strip())

    for candidate in direct_candidates:
        if not candidate:
            continue
        try:
            return json.loads(candidate)
        except Exception:
            pass

    fragment = _balanced_json_fragment(text, "[", "]")
    if fragment is None:
        return None
    try:
        return json.loads(fragment)
    except Exception:
        return None


def normalize_name(value, candidate_names):
    if value in candidate_names:
        return value

    candidate_names_by_str = {str(name): name for name in candidate_names}

    if isinstance(value, int):
        return candidate_names_by_str.get(str(value))

    if not isinstance(value, str):
        return None

    stripped = value.strip()
    if stripped in candidate_names:
        return stripped
    if stripped in candidate_names_by_str:
        return candidate_names_by_str[stripped]
    if stripped.lower().startswith("person "):
        return candidate_names_by_str.get(stripped.split(" ", 1)[1].strip())
    if stripped.lower().startswith("candidate "):
        return candidate_names_by_str.get(stripped.split(" ", 1)[1].strip())

    return None

def network_growth(n0, temperature=None, model='gpt-5-mini', environment=None, role='friends', method='llm', num_common_neighbors=True, cot=False, cot_config=None, er=False):

    if method == 'sbm':
        G = nx.stochastic_block_model([n0 // 2, n0 // 2], [[0.5, 0.1], [0.1, 0.5]])
   
        return [G], []

    else:
        if er:
            G = nx.erdos_renyi_graph(n0, 0.1, seed=0)
        else:
            G = nx.stochastic_block_model([n0 // 2, n0 // 2], [[0.5, 0.1], [0.1, 0.5]], seed=0)


    Gs = [G.copy()]
    results = []

    for t in G.nodes():

        if method == 'llm':
            result = select_neighbor(G, t, temperature, num_common_neighbors=num_common_neighbors, model=model, environment=environment, role=role, cot=cot, cot_config=cot_config)
            if result:
                v = result['name']
                G.add_edge(t, v)
                results.append(result)
        elif method == 'random':
            v = random.choice(list(set(G.nodes() - set(G.neighbors(t)))))
            G.add_edge(t, v)
            results.append({'name' : v, 'common_friends' : list(set(G.neighbors(v)) & set(G.neighbors(t))), 'reason' : 'random'})
        elif method == 'winner':
            v = None,
            max_common_friends = 0
            for u in G.nodes():
                if u not in G.neighbors(t) and u != t and len(set(G.neighbors(u)) & set(G.neighbors(t))) > max_common_friends:
                    v = u
                    max_common_friends = len(set(G.neighbors(u)) & set(G.neighbors(t)))

            G.add_edge(t, v)
            results.append({'name' : v, 'common_friends' : list(set(G.neighbors(v)) & set(G.neighbors(t))), 'reason' : 'winner'})

        Gs.append(G.copy())

    return Gs, results

def build_neighbor_request(G, t, environment, role, num_common_neighbors, cot, model, cot_config=None):
    candidate_profiles = []
    for v in G.nodes():
        if v != t and v not in G.neighbors(t):
            if num_common_neighbors:
                candidate_profiles.append({'name' : v, 'common_friends' : len(set(G.neighbors(v)) & set(G.neighbors(t)))})
            else:
                candidate_profiles.append({'name' : v, 'friends' : list(G.neighbors(v))})

    if cot: 
        output_format = f"""
    {{
        "reason" : reason for selecting the person,
        "name" : name of the person you selected
    }}
        """
    else:
        output_format = f"""
    {{
        "name" : name of the person you selected,
        "reason" : reason for selecting the person
    }}
        """

    candidate_names = [candidate['name'] for candidate in candidate_profiles]
    response_schema = build_response_schema(candidate_names)
    use_structured_output = not (model.startswith('Qwen/') and cot)
    allowed_names_json = json.dumps(candidate_names, ensure_ascii=False)

    prompt = f"""
    # Task
    {f'You are in a {environment}.' if environment else ''}Your task is to select a person to be {role} with.

    # Input
    The input is a list of dictionaries. 
    Your profile is given below after chevrons:
    <PROFILE>
    {json.dumps({'name' : t, 'friends' : list(G.neighbors(t))}, separators=(',', ':'))}
    </PROFILE>

    The list of candidate profiles is given below after chevrons:
    <PROFILES>
    {json.dumps(candidate_profiles, separators=(',', ':'))}
    </PROFILES>

    # Output
    The output should be given in JSON format with the following structure

    {output_format}

    # Notes
    * Return exactly one JSON object.
    * The "reason" value must be at most 10 words.
    * Do not list node IDs or friend IDs in "reason".
    * If your chat template enables thinking, keep reasoning in the thinking section.
    * After any thinking, write a line starting with FINAL_JSON: followed by exactly one JSON object.
    * Do not explain your reasoning outside the JSON object in the final answer.
    * Do not write markdown fences.
    * Do not write any text before or after the JSON object.
    * The value of "name" must be exactly one of these values: {allowed_names_json}
    * Do not rename the person.
    * Do not output labels such as "person 0", "Person 0", or "candidate 0".
    * The final answer must be exactly one JSON object and must not contain text after the JSON object.
    """   

    return {
        'prompt': prompt,
        'response_schema': response_schema if use_structured_output else None,
        'candidate_names': set(candidate_names),
    }


def parse_neighbor_response(ans, request):
    result = first_json_object(ans)
    recovered = False
    if not isinstance(result, dict) or 'name' not in result:
        result = recover_name_from_malformed_response(ans, request['candidate_names'])
        recovered = True
    normalized_name = normalize_name(result['name'], request['candidate_names'])
    if normalized_name is None:
        raise ValueError(f"Invalid candidate name: {result['name']}")
    result['name'] = normalized_name
    if recovered:
        result.setdefault('reason', 'Recovered from malformed JSON response')
        result['parse_recovered'] = True
    return result


def recover_name_from_malformed_response(ans, candidate_names):
    if ans is None:
        raise ValueError('Could not parse a valid JSON object with a name field.')

    text = str(ans)
    patterns = [
        r'"name"\s*:\s*(-?\d+)',
        r"'name'\s*:\s*(-?\d+)",
        r'"name"\s*:\s*"([^"\\]*(?:\\.[^"\\]*)*)"',
        r"'name'\s*:\s*'([^']*)'",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if not match:
            continue
        value = match.group(1)
        if re.fullmatch(r'-?\d+', value):
            value = int(value)
        normalized_name = normalize_name(value, candidate_names)
        if normalized_name is not None:
            print(f'Recovered candidate name from malformed JSON response: {normalized_name}')
            return {'name': normalized_name, 'reason': 'Recovered from malformed JSON response'}

    raise ValueError('Could not parse a valid JSON object with a name field.')


def print_llm_parse_error(e, ans, context='', max_chars=4000):
    prefix = f'LLM response parse error ({context})' if context else 'LLM response parse error'
    print(f'{prefix}: {type(e).__name__}: {e}')
    if ans is None:
        print('LLM raw output: <unavailable>')
        return

    text = str(ans)
    truncated = len(text) > max_chars
    preview = text[:max_chars]
    print('LLM raw output repr:', repr(preview))
    if truncated:
        print(f'LLM raw output truncated at {max_chars} of {len(text)} characters.')


def retry_cot_config(cot_config, attempt):
    if not cot_config:
        return cot_config
    config = dict(cot_config)
    if attempt <= 0:
        return config
    base_tokens = int(config.get('max_new_tokens', 1000))
    retry_max_new_tokens = int(globals().get('COT_RETRY_MAX_NEW_TOKENS', 32768))
    config['max_new_tokens'] = max(base_tokens, retry_max_new_tokens)
    if attempt >= 2:
        config['qwen_enable_thinking'] = False
    return config


def select_neighbor(G, t, temperature, model, environment, role, num_common_neighbors, cot, cot_config=None):
    request = build_neighbor_request(G, t, environment, role, num_common_neighbors, cot, model, cot_config)
    for i in range(3):
        ans = None
        try:
            attempt_cot_config = retry_cot_config(cot_config, i) if cot else cot_config
            if cot and attempt_cot_config and attempt_cot_config != cot_config:
                print(f'Retrying with max_new_tokens={attempt_cot_config["max_new_tokens"]}')
            ans = get_response(request['prompt'], temperature=temperature, system_prompt="You are a helpful assistant", model=model, response_schema=request['response_schema'], cot=cot, cot_config=attempt_cot_config)
            result = parse_neighbor_response(ans, request)
            if i > 0:
                result['retry_attempt'] = i + 1
                if attempt_cot_config and 'max_new_tokens' in attempt_cot_config:
                    result['retry_max_new_tokens'] = attempt_cot_config['max_new_tokens']
            print('NEW EDGE', result)
            return result 
        except Exception as e:
            print_llm_parse_error(e, ans, context=f'select_neighbor attempt={i + 1}, node={t}, model={model}')
    
def initialize_growth_state(n, temperature, experiment_record):
    if experiment_record['er']:
        G = nx.erdos_renyi_graph(n, 0.1, seed=0)
    else:
        G = nx.stochastic_block_model([n // 2, n // 2], [[0.5, 0.1], [0.1, 0.5]], seed=0)

    return {
        'n': n,
        'temperature': temperature,
        'temperature_label': 'default' if temperature is None else temperature,
        'model': experiment_record['model'],
        'environment': experiment_record['environment'],
        'role': experiment_record['role'],
        'num_common_neighbors': experiment_record['num_common_neighbors'],
        'cot': experiment_record['cot'],
        'cot_config': experiment_record.get('cot_config'),
        'er': experiment_record['er'],
        'outfile': experiment_record['outfile'],
        'metadata': experiment_record['metadata'],
        'G': G,
        'node_order': list(G.nodes()),
        't_idx': 0,
        'graphs': [G.copy()],
        'reasons': [],
    }


RESET_OUTFILES = set()


def checkpoint_filename(outfile, n, simulation, temperature_label):
    stem, ext = os.path.splitext(outfile)
    safe_temperature = str(temperature_label).replace(os.sep, '_')
    return f'{stem}.n{n}.sim{simulation}.temp{safe_temperature}.checkpoint.json'


def serialize_growth_state(state):
    temp = {
        'n' : state['n'],
        'temperature' : state['temperature_label'],
        'simulation' : state['simulation'],
        'graphs' : [nx.to_dict_of_lists(G) for G in state['graphs']],
        'reasons' : state['reasons'],
        'model' : state['model'],
        'environment' : state['environment'] if state['environment'] is not None else 'Baseline',
        'role' : state['role'],
        'num_common_neighbors' : state['num_common_neighbors'],
        'cot' : state['cot'],
        'er' : state['er'],
    }
    if state['metadata']:
        temp.update(state['metadata'])
    return temp


def save_growth_checkpoint(state):
    checkpoint = serialize_growth_state(state)
    checkpoint['t_idx'] = state['t_idx']
    checkpoint['node_order'] = state['node_order']
    checkpoint_path = checkpoint_filename(state['outfile'], state['n'], state['simulation'], state['temperature_label'])
    with open(checkpoint_path, 'w') as f:
        json.dump(checkpoint, f)
        f.flush()


def remove_growth_checkpoint(state):
    checkpoint_path = checkpoint_filename(state['outfile'], state['n'], state['simulation'], state['temperature_label'])
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)


def load_growth_checkpoint(experiment_record, n, simulation, temperature, temperature_label):
    checkpoint_path = checkpoint_filename(experiment_record['outfile'], n, simulation, temperature_label)
    if not os.path.exists(checkpoint_path):
        return None
    with open(checkpoint_path) as f:
        checkpoint = json.load(f)
    state = initialize_growth_state(n, temperature, experiment_record)
    state['simulation'] = simulation
    state['temperature_label'] = temperature_label
    state['node_order'] = checkpoint.get('node_order', state['node_order'])
    state['t_idx'] = checkpoint['t_idx']
    state['graphs'] = []
    for graph in checkpoint['graphs']:
        G = nx.Graph()
        for k, values in graph.items():
            k = int(k)
            G.add_node(k)
            for value in values:
                G.add_edge(k, value)
        state['graphs'].append(G)
    state['G'] = state['graphs'][-1].copy()
    state['reasons'] = checkpoint.get('reasons', [])
    print(f'Resuming checkpoint for n={n}, i={simulation}, temperature={temperature_label}, t_idx={state["t_idx"]}')
    return state


def maybe_reset_outfile(outfile):
    if not IGNORE_EXISTING_OUTPUTS:
        return
    if outfile in RESET_OUTFILES:
        return
    if os.path.exists(outfile):
        os.remove(outfile)
        print(f'Removed existing output file {outfile}')
    RESET_OUTFILES.add(outfile)


def pending_growth_states_for_experiment(experiment_record):
    parameters = experiment_record['parameters']
    temperatures = experiment_record['temperatures']
    outfile = experiment_record['outfile']
    os.makedirs(os.path.dirname(outfile), exist_ok=True)
    maybe_reset_outfile(outfile)

    saved_scenarios = set()
    if os.path.exists(outfile) and not IGNORE_EXISTING_OUTPUTS:
        with open(outfile) as f:
            lines = f.read().splitlines()
        for line in lines:
            scenario = json.loads(line)
            saved_scenarios.add((scenario['n'], scenario['simulation'], scenario['temperature']))

    print(f'Loaded {len(saved_scenarios)} completed simulations from {outfile}')
    states = []
    for n in range(parameters['n_min'], parameters['n_max'] + 1, parameters['n_step']):
        for i in range(parameters['num_simulations']):
            for temperature in temperatures:
                temperature_label = 'default' if temperature is None else temperature
                if (n, i, temperature_label) in saved_scenarios:
                    print(f'Skipping simulation for n={n}, i={i}, temperature={temperature_label}')
                    continue
                checkpoint_state = load_growth_checkpoint(experiment_record, n, i, temperature, temperature_label)
                if checkpoint_state is not None:
                    states.append(checkpoint_state)
                    continue
                print(f'Queueing simulation for n={n}, i={i}, temperature={temperature_label}, outfile={outfile}')
                state = initialize_growth_state(n, temperature, experiment_record)
                state['simulation'] = i
                states.append(state)
    return states


def advance_growth_state(state, result):
    t = state['node_order'][state['t_idx']]
    if result:
        state['G'].add_edge(t, result['name'])
        state['reasons'].append(result)
    state['graphs'].append(state['G'].copy())
    state['t_idx'] += 1
    save_growth_checkpoint(state)


def write_growth_state(state):
    temp = serialize_growth_state(state)
    with open(state['outfile'], 'a+') as f:
        f.write(json.dumps(temp) + '\n')
        f.flush()
    remove_growth_checkpoint(state)


def run_network_formation_experiments_batch(experiment_records):
    if not experiment_records:
        return

    model = experiment_records[0]['model']
    cot = experiment_records[0]['cot']
    cot_config = experiment_records[0].get('cot_config')
    active_states = []
    for experiment_record in experiment_records:
        active_states.extend(pending_growth_states_for_experiment(experiment_record))

    if not active_states:
        print(f'All batched simulations already completed for {model}. Skipping inference.')
        return

    print(f'Running {len(active_states)} batched simulations for {model}, cot={cot}')
    while active_states:
        requests_by_temperature = collections.defaultdict(list)
        for state in active_states:
            t = state['node_order'][state['t_idx']]
            print(f'Adding edge for node {t} in {state["metadata"]["experiment_name"]}, simulation={state["simulation"]}')
            request = build_neighbor_request(
                state['G'],
                t,
                state['environment'],
                state['role'],
                state['num_common_neighbors'],
                state['cot'],
                state['model'],
                state.get('cot_config'),
            )
            requests_by_temperature[state['temperature']].append((state, request))

        results_by_state_id = {}
        for temperature, batch_items in requests_by_temperature.items():
            remaining = list(batch_items)
            for attempt in range(3):
                if not remaining:
                    break

                prompts = [request['prompt'] for _, request in remaining]
                response_schemas = [request['response_schema'] for _, request in remaining]
                attempt_cot_config = retry_cot_config(cot_config, attempt) if cot else cot_config
                if cot and attempt_cot_config and attempt_cot_config != cot_config:
                    print(f'Retrying batch with max_new_tokens={attempt_cot_config["max_new_tokens"]}')
                answers = get_responses(
                    prompts,
                    model,
                    temperature=temperature,
                    system_prompt="You are a helpful assistant",
                    response_schemas=response_schemas,
                    cot=cot,
                    cot_config=attempt_cot_config,
                )
                if all(ans is None for ans in answers):
                    print(f'No local answers available for {model}; marking this batch step as missing.')
                    break

                next_remaining = []
                for (state, request), ans in zip(remaining, answers):
                    try:
                        result = parse_neighbor_response(ans, request)
                        if attempt > 0:
                            result['retry_attempt'] = attempt + 1
                            if attempt_cot_config and 'max_new_tokens' in attempt_cot_config:
                                result['retry_max_new_tokens'] = attempt_cot_config['max_new_tokens']
                        print('NEW EDGE', result)
                        results_by_state_id[id(state)] = result
                    except Exception as e:
                        print_llm_parse_error(
                            e,
                            ans,
                            context=(
                                f'batch attempt={attempt + 1}, '
                                f'experiment={state["metadata"]["experiment_name"]}, '
                                f'simulation={state["simulation"]}, '
                                f'node={state["node_order"][state["t_idx"]]}, '
                                f'model={state["model"]}, '
                                f'temperature={state["temperature_label"]}'
                            ),
                        )
                        next_remaining.append((state, request))
                remaining = next_remaining

            for state, _ in remaining:
                results_by_state_id[id(state)] = None

        next_active_states = []
        for state in active_states:
            advance_growth_state(state, results_by_state_id.get(id(state)))
            if state['t_idx'] >= len(state['node_order']):
                write_growth_state(state)
            else:
                next_active_states.append(state)
        active_states = next_active_states


def run_network_formation_experiment(n_min, n_max, n_step, num_simulations, outfile, temperatures=None, method='llm', model='gpt-5-mini', environment=None, role='friends', num_common_neighbors=True, cot=False, cot_config=None, er=False, metadata=None):
    """ Run the network formation experiment."""
    if temperatures is None:
        temperatures = [None]

    os.makedirs(os.path.dirname(outfile), exist_ok=True)
    maybe_reset_outfile(outfile)

    saved_scenarios = set()

    if os.path.exists(outfile) and not IGNORE_EXISTING_OUTPUTS:
        with open(outfile) as f:
            lines = f.read().splitlines()

        for line in lines:
            scenario = json.loads(line)
            saved_scenarios.add((scenario['n'], scenario['simulation'], scenario['temperature']))

    expected_scenarios = {
        (n, i, 'default' if temperature is None else temperature)
        for n in range(n_min, n_max + 1, n_step)
        for i in range(num_simulations)
        for temperature in temperatures
    }

    if not IGNORE_EXISTING_OUTPUTS and expected_scenarios.issubset(saved_scenarios):
        print(f'All simulations already completed for {outfile}. Skipping inference.')
        return

    print(f'Loaded {len(saved_scenarios)} completed simulations from {outfile}')

    f = open(outfile, 'a+')

    for n in range(n_min, n_max + 1, n_step):
        for i in range(num_simulations):
            for temperature in temperatures:
                temperature_label = 'default' if temperature is None else temperature
                if (n, i, temperature_label) in saved_scenarios:
                    print(f'Skipping simulation for n={n}, i={i}, temperature={temperature_label}')
                    continue
                print(f'Running simulation for n={n}, i={i}, temperature={temperature_label}')

                experiment_record = {
                    'model': model,
                    'outfile': outfile,
                    'environment': environment,
                    'role': role,
                    'num_common_neighbors': num_common_neighbors,
                    'cot': cot,
                    'cot_config': cot_config,
                    'er': er,
                    'metadata': metadata or {},
                }
                state = load_growth_checkpoint(experiment_record, n, i, temperature, temperature_label)
                if state is None:
                    state = initialize_growth_state(n, temperature, experiment_record)
                    state['simulation'] = i

                while state['t_idx'] < len(state['node_order']):
                    t = state['node_order'][state['t_idx']]
                    print(f'Adding edge for node {t}, simulation={state["simulation"]}')
                    result = select_neighbor(
                        state['G'],
                        t,
                        state['temperature'],
                        state['model'],
                        state['environment'],
                        state['role'],
                        state['num_common_neighbors'],
                        state['cot'],
                        state.get('cot_config'),
                    )
                    advance_growth_state(state, result)

                write_growth_state(state)

                if method != 'llm':
                    break

    f.close()

def draw_graph(G, ax, G0=None, use_netgraph=True, er=False):
    if er:
        group_1 = [n for n in G.nodes()]
        group_2 = []
    else:
        group_1 = [n for n in G.nodes() if n < len(G.nodes()) // 2]
        group_2 = [n for n in G.nodes() if n >= len(G.nodes()) // 2]
    
    if not G0:
        G0_edges = set()
    else:
        G0_edges = set(G0.edges())
    G_edges = set(G.edges()) - G0_edges

    G_group_1 = (set(nx.subgraph(G, group_1).edges()) & G_edges) - G0_edges
    G0_group_1 = (set(nx.subgraph(G0, group_1).edges()))
    G_group_2 = (set(nx.subgraph(G, group_2).edges()) & G_edges) - G0_edges
    G0_group_2 = set(nx.subgraph(G0, group_2).edges())
    G_between = G_edges - set(nx.subgraph(G, group_1).edges()) - set(nx.subgraph(G, group_2).edges()) - G0_edges
    G0_between = G0_edges - set(nx.subgraph(G0, group_1).edges()) - set(nx.subgraph(G0, group_2).edges())
    pos = nx.spring_layout(G)

    if not use_netgraph:
       
        node_color = ['#c0392b' if n in group_1 else '#2980b9' for n in G.nodes()]

        if not G0:
            nx.draw(G, pos, ax=ax, node_size=10, width=0.5, node_color=node_color, alpha=0.7, edge_color='#34495e')
        else:
            
            nx.draw_networkx_edges(G, pos, edgelist=G0_edges, width=0.5, alpha=0.5, edge_color='#34495e', ax=ax)
            nx.draw_networkx_edges(G, pos, edgelist=G_between, width=1.0, alpha=1, edge_color='#f1c40f', ax=ax)
            nx.draw_networkx_edges(G, pos, edgelist=G_group_1, width=2, alpha=1, edge_color='#c0392b', ax=ax)
            nx.draw_networkx_edges(G, pos, edgelist=G0_group_1, width=1.0, alpha=0.5, edge_color='#e74c3c', ax=ax)

            nx.draw_networkx_edges(G, pos, edgelist=G_group_2, width=2, alpha=1, edge_color='#2980b9', ax=ax)
            nx.draw_networkx_edges(G, pos, edgelist=G0_group_2, width=1.0, alpha=0.5, edge_color='#3498db', ax=ax)

            nx.draw_networkx_nodes(G, pos, nodelist=list(G.nodes()), node_size=10, node_color=node_color, alpha=0.7, ax=ax)
    else:
        if er:
            node2community = {i: 0 for i in G.nodes()}
        else:
            node2community = {i: 0 if i < len(G.nodes()) // 2 else 1 for i in G.nodes()}
        
        node_color = {i : '#c0392b' if node2community[i] == 0 else  '#2980b9' for i in G.nodes()}

        edge_color = {}
        edge_width = {}
        edge_alpha = {}
        for (u, v) in G.edges():
            if (u, v) in G_group_1:
                edge_color[u, v] = '#c0392b'
            elif (u, v) in G_group_2:
                edge_color[u, v] = '#2980b9'
            elif (u, v) in G0_group_1:
                edge_color[u, v] = '#e74c3c'
            elif (u, v) in G0_group_2:
                edge_color[u, v] = '#3498db'
            elif (u, v) in G_between:
                edge_color[u, v] = '#f1c40f'
            else:
                edge_color[u, v] = '#bdc3c7'

            if (u, v) in G_group_1 or (u, v) in G_group_2 or (u, v) in G_between:
                edge_width[u, v] = 2
                edge_alpha[u, v] = 1
            else:
                edge_width[u, v] = 1
                edge_alpha[u, v] = 0.5

        # netgraph.Graph(G, node_layout='community', node_color=node_color, node_layout_kwargs=dict(node_to_community=node2community), node_size=2.5, edge_color=edge_color, edge_layout='bundled', edge_layout_kwargs=dict(k=2000), ax=ax)
        netgraph.Graph(G, node_layout=pos, node_color=node_color, node_layout_kwargs=dict(node_to_community=node2community), node_size=2.5, edge_color=edge_color, edge_width=edge_width, edge_alpha=edge_alpha, ax=ax)


    ax.set_axis_off()

def prob_edge_within_community(G, G0, er=False):
    if er:
        group_1 = [n for n in G.nodes()]
        group_2 = []
    else:
        group_1 = [n for n in G.nodes() if n < len(G.nodes()) // 2]
        group_2 = [n for n in G.nodes() if n >= len(G.nodes()) // 2]
    
    G0_edges = set(G0.edges())

    G_edges = set(G.edges()) - G0_edges

    G_between = G_edges - set(nx.subgraph(G, group_1).edges()) - set(nx.subgraph(G, group_2).edges()) - G0_edges

    try:
        return 1 - len(G_between) / (1e-1 + len(G_edges))
    except:
        return 0

def analyze_experiments(filename, num_common_neighbors=True, er=False, sfx=''):
    os.makedirs('figures/principle_2', exist_ok=True)

    suffix = os.path.split(os.path.splitext(filename)[0])[-1]

    with open(filename) as f:
        lines = f.read().splitlines()

    data = []

    for line in lines:
        data.append(json.loads(line))

    transitivities = collections.defaultdict(list)
    algebraic_connectivities = collections.defaultdict(list)
    probabilities_of_edge_within_community = collections.defaultdict(list)
    # partition_qualitys = collections.defaultdict(list)

    final_graphs = collections.defaultdict(list)

    for d in data:
        Gs = []
        for i, graph in enumerate(d['graphs']):
            G = nx.Graph()

            for k, v in graph.items():
                k = int(k)
                G.add_node(k)
                for n in v:
                    G.add_edge(k, n)

            G.remove_edges_from(nx.selfloop_edges(G))

            if i > 0:
                print('new edge', set(G.edges()) - set(Gs[0].edges()))

            Gs.append(G)

        # fig, ax = plt.subplots(1, 4, figsize=(20, 5))

        # fig.suptitle(f'Temperature = {d["temperature"]}')

        # for i, t in enumerate([0, len(Gs) // 2, len(Gs) - 1]):
        #     G = Gs[t]
        #     ax[i].set_title(f'$t = {t}$')
        #     draw_graph(G, ax=ax[i], G0=Gs[0])

            # print(d['reasons'])

        if er:
            group_1 = [n for n in G.nodes()]
            group_2 = []
        else:
            group_1 = [n for n in G.nodes() if n < len(G.nodes()) // 2]
            group_2 = [n for n in G.nodes() if n >= len(G.nodes()) // 2]


        final_graphs[d['n'], d['temperature']].append((Gs[-1], Gs[0]))

        initial_transitivity = nx.transitivity(Gs[0])

        transitivity = [nx.transitivity(G) - initial_transitivity for G in Gs]

        algebraic_connectivity = [nx.algebraic_connectivity(G) for G in Gs]

        probability_of_edge_within_community = [prob_edge_within_community(G, Gs[0], er=er) for G in Gs[1:]]

        # partition_quality = [nx.community.partition_quality(G, communities)[0] for G in Gs]

        # ax[-1].set_title('Metrics')
        # ax[-1].plot(transitivity, label='Marginal Transitivity', color='#c0392b')

        # ax_y = ax[-1].twinx()

        # ax_y.plot(algebraic_connectivity, label='Algebraic Connectivity', color='#2980b9')
        # ax[-1].set_xlabel('t')
        # ax[-1].set_ylabel('Transitivity', color='#c0392b')
        # ax_y.set_ylabel('Algebraic Connectivity', color='#2980b9')

        transitivities[d['n'], d['temperature']].append(transitivity)
        algebraic_connectivities[d['n'], d['temperature']].append(algebraic_connectivity)
        probabilities_of_edge_within_community[d['n'], d['temperature']].append(probability_of_edge_within_community)
        # partition_qualitys[d['n'], d['temperature']].append(partition_quality)

        # fig.tight_layout()
        # fig.savefig(f'figures/principle_2/{suffix}_{d["n"]}_{d["simulation"]}_{d["temperature"]}{"_neighbors" if not num_common_neighbors else ""}.pdf')

    palette = ['#e67e22', '#f1c40f', '#7f8c8d', '#c0392b', '#2980b9', '#34495e']


    # fig, ax = plt.subplots(4, len(transitivities), figsize=(5 * len(transitivities), 10), squeeze=False, sharey='row')

    if er:
        fig_final, ax_final = plt.subplots(1, len(final_graphs) + 1, figsize=(5 * (1 + len(final_graphs)), 5), squeeze=False)

        ax_final[0, -1].spines[['right', 'top']].set_visible(False)
    else:

        fig_final, ax_final = plt.subplots(1, len(final_graphs) + 2, figsize=(5 * (2 + len(final_graphs)), 5), squeeze=False, gridspec_kw={'width_ratios': [1] * len(final_graphs) + [0.5, 0.5]})

        ax_final[0, -1].spines[['right', 'top']].set_visible(False)
        ax_final[0, -2].spines[['right', 'top']].set_visible(False)



    for i, (k, v) in enumerate(sorted(final_graphs.items())):
        G, G0 = v[0]
        draw_graph(G, ax=ax_final[0, i], G0=G0, er=er)

        ax_final[0, i].set_title(f'Temperature = {k[1]}')


    if er:
        ax_final[0, -1].set_ylabel('Marginal Transitivity')
        ax_final[0, -1].set_xticks([])

    else:
        ax_final[0, -2].set_ylabel('Marginal Transitivity')
        ax_final[0, -2].set_xticks([])

            
        ax_final[0, -1].set_ylabel('Pr. Edge w Community')
        ax_final[0, -1].set_xticks([])

    for i, (k, c) in enumerate(zip(sorted(transitivities.keys()), palette)):
        v = transitivities[k]
        v = np.array(v)

        mean = v.mean(axis=0)
        std = v.std(axis=0)

        ci = 1.96 * std / np.sqrt(len(v))

        if er:
            ax_final[0, -1].bar(i, mean[-1], color=palette[i], alpha=0.6, label='Temp = ' + str(k[1]))
            ax_final[0, -1].errorbar(i, mean[-1], yerr=ci[-1], color='black', alpha=1)
        else:
            ax_final[0, -2].bar(i, mean[-1], color=palette[i], alpha=0.6, label='Temp = ' + str(k[1]))
            ax_final[0, -2].errorbar(i, mean[-1], yerr=ci[-1], color='black', alpha=1)


    if not er:
        for i, (k, c) in enumerate(zip(sorted(probabilities_of_edge_within_community.keys()), palette)):
            v = probabilities_of_edge_within_community[k]
            v = np.array(v)

            mean = v.mean(axis=0)
            std = v.std(axis=0)

            ci = 1.96 * std / np.sqrt(len(v))

            
            ax_final[0, -1].bar(i, mean[-1], color=palette[i], alpha=0.5, label='Temp = ' + str(k[1]))
            ax_final[0, -1].errorbar(i, mean[-1], yerr=ci[-1], color='black', alpha=0.5)



    # Null models
    transitivities_null = { 'random' : collections.defaultdict(list), 'winner' : collections.defaultdict(list), 'sbm' : collections.defaultdict(list) }
    algebraic_connectivities_null = { 'random' : collections.defaultdict(list), 'winner' : collections.defaultdict(list), 'sbm' : collections.defaultdict(list) }
    probabilities_of_edge_within_community_null = { 'random' : collections.defaultdict(list), 'winner' : collections.defaultdict(list), 'sbm' : collections.defaultdict(list) }

    for d in data:
        for method in ['random']:
            null_temperature = None if d['temperature'] == 'default' else d['temperature']
            Gs, _ = network_growth(d['n'], null_temperature, method=method, model='gpt-5-mini', environment=None, role='friends', num_common_neighbors=num_common_neighbors, cot=False, er=er)

            initial_transitivity = nx.transitivity(Gs[0])

            transitivity = [nx.transitivity(G) - initial_transitivity for G in Gs]

            transitivities_null[method][d['n'], d['temperature']].append(transitivity)


            algebraic_connectivity = [nx.algebraic_connectivity(G) for G in Gs]

            algebraic_connectivities_null[method][d['n'], d['temperature']].append(algebraic_connectivity)

            if er:
                group_1 = [n for n in Gs[0].nodes()]
                group_2 = []
            else:
                group_1 = [n for n in G.nodes() if n < len(G.nodes()) // 2]
                group_2 = [n for n in G.nodes() if n >= len(G.nodes()) // 2]

            communities = [group_1, group_2]

            probability_of_edge_within_community = [prob_edge_within_community(G, Gs[0], er=er) for G in Gs[1:]]

            probabilities_of_edge_within_community_null[method][d['n'], d['temperature']].append(probability_of_edge_within_community)

    

    for j, method in enumerate(['random']):
        for i, (k, v) in enumerate(transitivities_null[method].items()):
            v = np.array(v)

            mean = v.mean(axis=0)
            std = v.std(axis=0)

            ci = 1.96 * std / np.sqrt(len(v))

            if i == 0:

                if method == 'random':
                    transitivity_temp = mean.mean()
                    print('Transitivity null: ', transitivity_temp)

        
            if i == 0:
                if er:
                    ax_final[0, -1].bar(j + 3, mean[-1], color=palette[j+3], alpha=0.6, label=method.capitalize())
                    ax_final[0, -1].errorbar(j + 3, mean[-1], yerr=ci[-1], color='black', alpha=1)
                else:
                    ax_final[0, -2].bar(j + 3, mean[-1], color=palette[j+3], alpha=0.6, label=method.capitalize())
                    ax_final[0, -2].errorbar(j + 3, mean[-1], yerr=ci[-1], color='black', alpha=1)
            

            print('Transitivity T-test', k, method, scipy.stats.ttest_ind([x[-1] for x in transitivities[k]], [x[-1] for x in transitivities_null[method][k]], equal_var=False))

        for i, (k, v) in enumerate(algebraic_connectivities_null[method].items()):
            v = np.array(v)

            mean = v.mean(axis=0)
            std = v.std(axis=0)

            ci = 1.96 * std / np.sqrt(len(v))

            # print('Algebraic Connectivity T-test', k, method, scipy.stats.ttest_ind([x[-1] for x in algebraic_connectivities[k]], [x[-1] for x in algebraic_connectivities_null[method][k]], equal_var=False))

        for i, (k, v) in enumerate(probabilities_of_edge_within_community_null[method].items()):
            

            v = np.array(v)

            mean = v.mean(axis=0)
            std = v.std(axis=0)

            ci = 1.96 * std / np.sqrt(len(v))

            # ax[1, i].plot(mean, color='#c0392b' if method == 'random' else '#34495e', linestyle='--' if method == 'random' else ':', label=method.capitalize())
            # ax[1, i].fill_between(np.arange(len(mean)), mean - ci, mean + ci, alpha=0.2, color='#c0392b' if method == 'random' else '#34495e')
            if i == 0:
                # ax_combined[0, 2].plot(mean, color='#c0392b' if method == 'random' else '#34495e', linestyle='--' if method == 'random' else ':')
                # ax_combined[0, 2].fill_between(np.arange(len(mean)), mean - ci, mean + ci, alpha=0.2, color='#c0392b' if method == 'random' else '#34495e', hatch='||')

                if method == 'random':
                    probability_temp = mean.mean()
                    print('Probability null: ', probability_temp)

            if i == 0:
                if not er:
                    ax_final[0, -1].bar(j + 3, mean[-1], color=palette[j+3], alpha=0.6, label=method.capitalize())
                    ax_final[0, -1].errorbar(j + 3, mean[-1], yerr=ci[-1], color='black', alpha=1)


            print('Probability of edge within community T-test', k, method, scipy.stats.ttest_ind([x[-1] for x in probabilities_of_edge_within_community[k]], [x[-1] for x in probabilities_of_edge_within_community_null[method][k]], equal_var=False))

        # for i, (k, v) in enumerate(probabilities_of_edge_within_community_null[method].items()):
        #     v = np.array(v)

        #     mean = v.mean(axis=0)
        #     std = v.std(axis=0)

        #     ci = 1.96 * std / np.sqrt(len(v))

        #     ax[1, i].plot(mean, color='#c0392b' if method == 'random' else '#34495e', linestyle='--' if method == 'random' else ':', label=method.capitalize())
        #     ax[1, i].fill_between(np.arange(len(mean)), mean - ci, mean + ci, alpha=0.2, color='#c0392b' if method == 'random' else '#34495e')
        #     if i == 0:
        #         ax_combined[0, 3].plot(mean, color='#c0392b' if method == 'random' else '#34495e', linestyle='--' if method == 'random' else ':')
        #         ax_combined[0, 3].fill_between(np.arange(len(mean)), mean - ci, mean + ci, alpha=0.2, color='#c0392b' if method == 'random' else '#34495e', hatch='||')

        #     if i == 0:
        #         ax_final[0, -2].bar(j + 3, mean[-1], color=palette[j+3], alpha=0.6, label=method.capitalize())
        #         ax_final[0, -2].errorbar(j + 3, mean[-1], yerr=ci[-1], color='black', alpha=1)

        #     print('Partition Quality T-test', k, method, scipy.stats.ttest_ind([x[-1] for x in partition_qualitys[k]], [x[-1] for x in partition_qualitys_null[method][k]], equal_var=False))

    ax_final[0, -1].legend(bbox_to_anchor=(1, 0.5), loc='center left', frameon=False)

    # ax[0, 0].legend(loc='upper left')
    # ax[1, 0].legend(loc='upper left')

    # ax_combined[0, 0].legend(loc='upper left')
    # ax_combined[0, 1].legend(loc='upper left')

    # fig_combined.tight_layout()

    # fig.tight_layout()

    # fig.savefig(f'figures/principle_2/{suffix}_overall{"_neighbors" if not num_common_neighbors else ""}.pdf')

    # fig_combined.savefig(f'figures/principle_2/{suffix}_overall_combined{"_neighbors" if not num_common_neighbors else ""}.pdf')
     

    fig_final.tight_layout()

    fig_final.savefig(f'figures/principle_2/{suffix}_final{"_neighbors" if not num_common_neighbors else ""}{sfx}.pdf', bbox_inches='tight')

    return transitivity_temp, probability_temp

def get_table(filenames, sfx='', environments=True, transitivity_null=-1, probability_null=-1, er=False):
    os.makedirs('figures', exist_ok=True)
    os.makedirs('tables', exist_ok=True)

    records = []

    num_graphs = 0

    for filename in filenames:
        print(filename)
        suffix = os.path.split(os.path.splitext(filename)[0])[-1]
        suffix = suffix.split('+')

        if len(suffix) == 3:
            model = suffix[-2]
            environment = suffix[-1]
        elif len(suffix) == 2:
            model = suffix[-1]
            environment = 'Baseline'
        else:
            model = suffix[-1]
            environment = 'Baseline'

        with open(filename) as f:
            lines = f.read().splitlines()

        data = []

        for line in lines:
            data.append(json.loads(line))

        for d in data:
            if 'model' in d:
                model = str(d['model']).replace('/', '-')
                if d.get('cot') and not model.endswith('_cot'):
                    model = f'{model}_cot'
                environment = d.get('environment', 'Baseline')
                if environment is None:
                    environment = 'Baseline'
                if d.get('cot') and environment != 'Baseline' and not str(environment).endswith('_cot'):
                    environment = f'{environment}_cot'

            Gs = []

            for i, graph in enumerate(d['graphs']):
                G = nx.Graph()

                for k, v in graph.items():
                    k = int(k)
                    G.add_node(k)
                    for n in v:
                        G.add_edge(k, n)

                if i == 0:
                    top_common_neighbors = np.zeros(len(G.nodes()))
                    total = 0
                else:
                    new_edge =  set(G.edges()) - set(Gs[-1].edges())
                    if len(new_edge) == 0:
                        continue

                    new_edge = new_edge.pop()

                    u, v = new_edge

                    common_neighbors_u = [len(set(G.neighbors(u)) & set(G.neighbors(n))) for n in G.nodes() if n != u]

                    # find what position the new edge is in the sorted list of common neighbors
                    try:
                        pos = np.argsort(common_neighbors_u)[::-1].tolist().index(v)
                        top_common_neighbors[pos] += 1
                        total += 1
                    except:
                        pass

                G.remove_edges_from(nx.selfloop_edges(G))

                Gs.append(G)

            top_common_neighbors /= total

            top_common_neighbors = np.cumsum(top_common_neighbors)
            top_common_neighbors = np.insert(top_common_neighbors, 0, 0)


            initial_transitivity = nx.transitivity(Gs[0])
            final_transitivity = nx.transitivity(Gs[-1])
            marginal_transitivity = final_transitivity - initial_transitivity
            final_probability_of_edge_within_community = prob_edge_within_community(Gs[-1], Gs[0], er=er)

            record = {
                'Model' : model,
                'Environment' : environment,
                'Temperature' : d['temperature'],
                'Marginal Transitivity' : marginal_transitivity,
                # 'Algebraic Connectivity' : final_algebraic_connectivity,
                'Prob. of Edge within Community' : final_probability_of_edge_within_community,
                'Probability of Connecting to Top-$k$' : top_common_neighbors,
                'Top-$k$' : np.arange(0, len(top_common_neighbors)) / len(top_common_neighbors)
            }

            records.append(record)

        

    df = pd.DataFrame(records)

    rename_models = {
        'gpt-5-nano' : 'GPT-5 Nano',
        'gpt-5-mini' : 'GPT-5 Mini',
        'Qwen-Qwen3.5-4B' : 'Qwen 3.5 4B',
        'Qwen-Qwen3.5-2B' : 'Qwen 3.5 2B',
        'Qwen-Qwen3.5-0.8B' : 'Qwen 3.5 0.8B',
        'gpt-5-nano_cot' : 'GPT-5 Nano',
        'gpt-5-mini_cot' : 'GPT-5 Mini',
        'Qwen-Qwen3.5-4B_cot' : 'Qwen 3.5 4B',
        'Qwen-Qwen3.5-2B_cot' : 'Qwen 3.5 2B',
        'Qwen-Qwen3.5-0.8B_cot' : 'Qwen 3.5 0.8B'
    }

    rename_env = {
        'school' : 'School',
        'work' : 'Work',
        'community' : 'Community',
        'school_cot' : 'School',
        'work_cot' : 'Work',
        'community_cot' : 'Community',
    }


    ncols = 2 + int(environments)

    df['Model'] = df['Model'].apply(lambda x: rename_models.get(x, x))
    df['Environment'] = df['Environment'].apply(lambda x: rename_env.get(x, x))

    baseline_model_key = BASELINE_MODEL.replace('/', '-')
    baseline_model = rename_models.get(baseline_model_key, baseline_model_key)
    default_temperature = df[df['Temperature'].notna()]['Temperature'].iloc[0]

    df_model = df.query('Environment == "Baseline" and Temperature == @default_temperature')
    df_environment = df.query('Model == @baseline_model')

    df_temperature = df.query('Model == @baseline_model and Environment == "Baseline"')

    if er:
        fig, ax = plt.subplots(1, ncols, figsize=(5 * ncols, 5), squeeze=False)
    else:
        fig, ax = plt.subplots(2, ncols, figsize=(5 * ncols, 10))
    
    sc_model = sns.barplot(data=df_model, y='Marginal Transitivity', x='Model', ax=ax[0, 0], palette=['#e67e22', '#f1c40f', '#3498db', '#7f8c8d', '#c0392b', '#34495e', '#2980b9'])

    sc_temperature = sns.barplot(data=df_temperature, y='Marginal Transitivity', x='Temperature', ax=ax[0, 1], palette=['#e67e22', '#f1c40f', '#3498db', '#7f8c8d', '#c0392b', '#34495e', '#2980b9'])
    
    ax[0, 0].set_ylabel('$D$', fontsize=MEDIUM_SIZE)

    


    if not er:
        ax[0, 0].set_xticks([])
        ax[0, 1].set_xticks([])
    
        ax[0, 0].set_xlabel('')
        ax[0, 1].set_xlabel('')

        ax[0, 1].get_yaxis().set_visible(False)

    if er:
        ax[0, 0].set_ylim(0, 0.25)
        ax[0, 1].set_ylim(0, 0.25)
    else:
        ax[0, 0].set_ylim(0, 0.1)
        ax[0, 1].set_ylim(0, 0.1)



    if environments:
        sc_environment = sns.barplot(data=df_environment, y='Marginal Transitivity', x='Environment', ax=ax[0, ncols-1], palette=['#e67e22', '#f1c40f', '#3498db', '#7f8c8d', '#c0392b', '#34495e', '#2980b9'])
        ax[0, 2].get_yaxis().set_visible(False)
        ax[0, 2].set_xlabel('')
        if er:
            ax[0, 2].set_ylim(0, 0.25)
        else:
            ax[0, 2].set_ylim(0, 0.1)
            ax[0, 2].set_xticks([])


    if not er:
        sc_model = sns.barplot(data=df_model, y='Prob. of Edge within Community', x='Model', ax=ax[1, 0], palette=['#e67e22', '#f1c40f', '#3498db', '#7f8c8d', '#c0392b', '#34495e', '#2980b9'])
        sc_temperature = sns.barplot(data=df_temperature, y='Prob. of Edge within Community', x='Temperature', ax=ax[1, 1], palette=['#e67e22', '#f1c40f', '#3498db', '#7f8c8d', '#c0392b', '#34495e', '#2980b9'])

    if transitivity_null != -1:
        ax[0, 0].axhline(y=transitivity_null, color='black', linestyle='--')
        ax[0, 1].axhline(y=transitivity_null, color='black', linestyle='--')
        ax[0, 2].axhline(y=transitivity_null, color='black', linestyle='--')

    if probability_null != -1 and not er:
        ax[1, 0].axhline(y=probability_null, color='black', linestyle='--')
        ax[1, 1].axhline(y=probability_null, color='black', linestyle='--')
        ax[1, 2].axhline(y=probability_null, color='black', linestyle='--')

    sc_model.set_xticklabels(sc_model.get_xticklabels(), rotation=90)
    sc_temperature.set_xticklabels(sc_temperature.get_xticklabels(), rotation=90)

    if er and environments:
        sc_environment.set_xticklabels(sc_environment.get_xticklabels(), rotation=90)
        
    ax[0, 0].set_title('Model')
    ax[0, 1].set_title('Temperature')

    if not er:
        ax[1, 0].set_ylabel('$\\hat p$', fontsize=MEDIUM_SIZE)
        ax[1, 1].set_ylabel('')

        ax[1, 0].set_xlabel('')
        ax[1, 1].set_xlabel('')


        ax[1, 1].get_yaxis().set_visible(False)

        ax[1, 0].set_ylim(0, 1)
        ax[1, 1].set_ylim(0, 1)

        ax[1, 0].xaxis.label.set_size(MEDIUM_SIZE)
        ax[1, 1].xaxis.label.set_size(MEDIUM_SIZE)
        
    if environments and not er:
        sc_environment = sns.barplot(data=df_environment, y='Prob. of Edge within Community', x='Environment', ax=ax[1, 2], palette=['#e67e22', '#f1c40f', '#3498db', '#7f8c8d', '#c0392b', '#34495e', '#2980b9'])
        sc_environment.set_xticklabels(sc_environment.get_xticklabels(), rotation=90)
        ax[1, 2].set_ylim(0, 1)
        ax[1, 2].get_yaxis().set_visible(False)
        ax[1, 2].set_xlabel('')
        ax[1, 2].set_ylabel('')
        ax[1, 2].spines[['right', 'top']].set_visible(False)
        ax[1, 2].xaxis.label.set_size(MEDIUM_SIZE)
        

    ax[0, 0].spines[['right', 'top']].set_visible(False)
    ax[0, 1].spines[['right', 'top']].set_visible(False)
    ax[0, 2].spines[['right', 'top']].set_visible(False)
    ax[0, 2].set_title('Environment')


    ax[0, 1].set_ylabel('')
    ax[0, 2].set_ylabel('')
    ax[0, 0].set_xlabel('')
    ax[0, 1].set_xlabel('')
    ax[0, 2].set_xlabel('')


    if not er:
        ax[1, 0].spines[['right', 'top']].set_visible(False)
        ax[1, 1].spines[['right', 'top']].set_visible(False)

    for i in range(len(ax)):
        for j in range(len(ax[i])):
            ax[i, j].tick_params(axis='both', which='major', labelsize=MEDIUM_SIZE)
            ax[i, j].yaxis.label.set_size(MEDIUM_SIZE)
            ax[i, j].xaxis.label.set_size(MEDIUM_SIZE)
            ax[i, j].title.set_size(MEDIUM_SIZE)

    fig.savefig(f'figures/triadic_closure{sfx}.pdf', bbox_inches='tight')

    fig, ax = plt.subplots(1, ncols, figsize=(5 * ncols, 5))

    palette=['#e67e22', '#f1c40f', '#3498db', '#7f8c8d', '#c0392b', '#34495e', '#2980b9']

    fig.suptitle('Probability of Connecting to Top-$k$ Common Neighbors', fontsize=SMALL_SIZE)

    breakpoints_arr = [('top', np.array([0.1, 0.2, 0.3, 0.4, 0.5])), ('all', np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99]))]

    for label, breakpoints in breakpoints_arr:
        breakpoint_max = np.max(breakpoints)
        breakpoint_min = np.min(breakpoints)

        for i in range(len(ax)):
            ax[i].plot([0, 100 * breakpoint_max], [0, breakpoint_max], color='black', linestyle='--')
            ax[i].set_xlim(100 * breakpoint_min, 100 * breakpoint_max)

        for i, model in enumerate(df_model['Model'].unique()):
            temp = df_model[df_model['Model'] == model]

            n = len(temp['Top-$k$'].values[0])

            indices = np.array([int(x * n) for x in breakpoints])

            color = palette[i]
            linewidth = 1

            if model == baseline_model and df_model['Environment'].values[i] == 'Baseline' and df_model['Temperature'].values[i] == default_temperature:
                color = '#34495e'
                linewidth = 3

            ax[0].plot(100 * temp['Top-$k$'].values[0][indices], temp['Probability of Connecting to Top-$k$'].values.mean(0)[indices], label=model, color=color, linewidth=linewidth, marker='x')

        

        ax[0].set_title('Model')
        ax[0].set_xlabel('Top-$k$ (%)')
        # ax[0].set_xscale('log')
        # ax[0].set_yscale('log')
        ax[0].set_ylabel('')
        
        ax[0].legend(fontsize=0.7*SMALL_SIZE, loc='upper left')

        for i, temperature in enumerate(df_temperature['Temperature'].unique()):
            temp = df_temperature[df_temperature['Temperature'] == temperature]

            n = len(temp['Top-$k$'].values[0])

            indices = np.array([int(x * n) for x in breakpoints])

            color = palette[i]
            linewidth = 1

            if temperature == default_temperature and df_temperature['Model'].values[i] == baseline_model and df_temperature['Environment'].values[i] == 'Baseline':
                color = '#34495e'
                linewidth = 3
                

            ax[1].plot(100 * temp['Top-$k$'].values[0][indices], temp['Probability of Connecting to Top-$k$'].values.mean(0)[indices], label=f'{temperature}', color=color, linewidth=linewidth, marker='x')


        ax[1].set_title('Temperature')
        ax[1].set_xlabel('Top-$k$ (%)')
        # ax[1].set_xscale('log')
        # ax[1].set_yscale('log')
        ax[1].set_ylabel('')

        ax[1].legend(fontsize=0.7*SMALL_SIZE, loc='upper left')

        if environments:

            for i, environment in enumerate(df_environment['Environment'].unique()):
                temp = df_environment[df_environment['Environment'] == environment]

                n = len(temp['Top-$k$'].values[0])

                indices = np.array([int(x * n) for x in breakpoints])

                color = palette[i]
                linewidth = 1

                if environment == 'Baseline' and df_environment['Model'].values[i] == baseline_model and df_environment['Temperature'].values[i] == default_temperature:
                    color = '#34495e'
                    linewidth = 3        
                    ax[2].plot(100 * temp['Top-$k$'].values[0][indices], df_model[df_model['Model'] == baseline_model]['Probability of Connecting to Top-$k$'].values.mean(0)[indices], label=baseline_model, color=color, linewidth=linewidth, marker='x')

                else:
                    ax[2].plot(100 * temp['Top-$k$'].values[0][indices], temp['Probability of Connecting to Top-$k$'].values.mean(0)[indices], label=environment, color=color, linewidth=linewidth, marker='x')


            ax[2].set_title('Environment')
            ax[2].set_xlabel('Top-$k$ (%)')
        
            ax[2].set_ylabel('')
            ax[2].legend(fontsize=0.7*SMALL_SIZE, loc='upper left')
            ax[2].set_ylim(0, 1)
            ax[2].get_yaxis().set_visible(False)
            ax[2].spines[['right', 'top']].set_visible(False)


        ax[0].set_ylim(0, 1)
        ax[1].set_ylim(0, 1)

        # hide y axis numbers
        ax[1].get_yaxis().set_visible(False)

        ax[0].spines[['right', 'top']].set_visible(False)
        ax[1].spines[['right', 'top']].set_visible(False)

        fig.tight_layout()

                        
        fig.savefig(f'figures/top_kcommon{label}_{sfx}.pdf', bbox_inches='tight')




## Network Formation Experiments


### Configured Triadic-Closure Experiments

このセルは `EXPERIMENTS` を順に読み、各辞書の設定に従って実験を実行します。出力ファイル名は `name` から `OUTPUT_DIR / principle_2_{name}.jsonl` として生成し、`summary_group` ごとに解析します。


In [ ]:
# Calibrate the first CoT retry token budget from preliminary raw outputs.
CALIBRATE_COT_RETRY_MAX_NEW_TOKENS = True
COT_CALIBRATION_SAMPLE_SIZE = 20
COT_CALIBRATION_MAX_NEW_TOKENS = 65536
COT_CALIBRATION_PERCENTILE = 0.90
COT_CALIBRATION_MARGIN = 1.5
COT_RETRY_TOKEN_BUCKETS = [8192, 16384, 32768, 65536]
COT_CALIBRATION_SEED = 0
COT_CALIBRATION_FILE = OUTPUT_DIR / 'principle_2_cot_budget_calibration.json'


def calibration_experiment_record(experiment):
    environment_role = experiment.get('environment')
    if environment_role is None:
        environment = None
        role = 'friends'
    else:
        environment, role = environment_role

    return {
        'experiment': experiment,
        'name': experiment['name'],
        'model': experiment['model'],
        'outfile': str(OUTPUT_DIR / f"principle_2_{experiment['name']}.jsonl"),
        'parameters': experiment['parameters'],
        'temperatures': experiment.get('temperatures', DEFAULT_TEMPERATURES),
        'environment': environment,
        'role': role,
        'method': experiment.get('method', 'llm'),
        'num_common_neighbors': experiment.get('num_common_neighbors', False),
        'cot': experiment.get('COT', False),
        'cot_config': experiment.get('cot_config'),
        'er': experiment.get('er', False),
        'summary_group': experiment.get('summary_group', 'sbm'),
        'metadata': {
            'experiment_name': experiment['name'],
            'summary_group': experiment.get('summary_group', 'sbm'),
            'model': experiment['model'],
            'environment': environment if environment is not None else 'Baseline',
            'role': role,
            'num_common_neighbors': experiment.get('num_common_neighbors', False),
            'cot': experiment.get('COT', False),
            'er': experiment.get('er', False),
        },
    }


def build_cot_calibration_requests(experiment, sample_size=COT_CALIBRATION_SAMPLE_SIZE):
    record = calibration_experiment_record(experiment)
    n = record['parameters']['n_max']
    temperature = record['temperatures'][0]
    state = initialize_growth_state(n, temperature, record)
    rng = random.Random(COT_CALIBRATION_SEED)
    nodes = list(state['node_order'])
    rng.shuffle(nodes)
    nodes = nodes[:sample_size]

    requests = []
    for node in nodes:
        request = build_neighbor_request(
            state['G'],
            node,
            state['environment'],
            state['role'],
            state['num_common_neighbors'],
            True,
            state['model'],
            state.get('cot_config'),
        )
        requests.append((node, request))
    return record, requests


def run_cot_budget_calibration():
    cot_experiments = [
        experiment for experiment in EXPERIMENTS
        if experiment.get('run', True)
        and experiment.get('COT', False)
        and experiment['model'].startswith('Qwen/')
    ]
    if not cot_experiments:
        print('No Qwen CoT experiments found; keeping COT_RETRY_MAX_NEW_TOKENS =', COT_RETRY_MAX_NEW_TOKENS)
        return None

    if COT_CALIBRATION_FILE.exists() and not IGNORE_EXISTING_OUTPUTS:
        with open(COT_CALIBRATION_FILE) as f:
            summary = json.load(f)
        globals()['COT_RETRY_MAX_NEW_TOKENS'] = int(summary['selected_max_new_tokens'])
        print('Loaded CoT retry budget calibration:', COT_RETRY_MAX_NEW_TOKENS)
        return summary

    experiment = cot_experiments[0]
    record, request_items = build_cot_calibration_requests(experiment)
    prompts = [request['prompt'] for _, request in request_items]
    response_schemas = [request['response_schema'] for _, request in request_items]
    calibration_cot_config = dict(record.get('cot_config') or DEFAULT_COT_CONFIG)
    calibration_cot_config.update({
        'max_new_tokens': COT_CALIBRATION_MAX_NEW_TOKENS,
        'qwen_enable_thinking': True,
        'strip_qwen_thinking': False,
    })

    print(
        f'Running {len(prompts)} CoT budget calibration generations for {record["model"]} '
        f'with max_new_tokens={COT_CALIBRATION_MAX_NEW_TOKENS}'
    )
    outputs = get_responses(
        prompts,
        record['model'],
        temperature=record['temperatures'][0],
        system_prompt='You are a helpful assistant',
        response_schemas=response_schemas,
        cot=True,
        cot_config=calibration_cot_config,
    )

    parse_successes = []
    for (node, request), output in zip(request_items, outputs):
        try:
            parse_neighbor_response(output, request)
            parse_successes.append(True)
        except Exception:
            parse_successes.append(False)

    tokenizer = None
    try:
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(record['model'], trust_remote_code=True)
    except Exception as e:
        print(f'Could not load tokenizer for calibration token counts; using character estimate: {e}')

    summary = summarize_generation_budget_calibration(
        outputs,
        tokenizer=tokenizer,
        parse_successes=parse_successes,
        percentile=COT_CALIBRATION_PERCENTILE,
        margin=COT_CALIBRATION_MARGIN,
        buckets=COT_RETRY_TOKEN_BUCKETS,
        minimum_budget=min(COT_RETRY_TOKEN_BUCKETS),
    )
    summary.update({
        'model': record['model'],
        'experiment_name': record['name'],
        'sample_size': len(outputs),
        'calibration_max_new_tokens': COT_CALIBRATION_MAX_NEW_TOKENS,
        'sampled_nodes': [node for node, _ in request_items],
    })

    globals()['COT_RETRY_MAX_NEW_TOKENS'] = int(summary['selected_max_new_tokens'])
    with open(COT_CALIBRATION_FILE, 'w') as f:
        json.dump(summary, f, indent=2)
    print('Selected COT_RETRY_MAX_NEW_TOKENS =', COT_RETRY_MAX_NEW_TOKENS)
    print('Calibration summary saved to', COT_CALIBRATION_FILE)
    return summary


if CALIBRATE_COT_RETRY_MAX_NEW_TOKENS and RUN_EXPERIMENTS:
    COT_BUDGET_CALIBRATION_SUMMARY = run_cot_budget_calibration()
else:
    print('CoT retry budget calibration skipped; using COT_RETRY_MAX_NEW_TOKENS =', COT_RETRY_MAX_NEW_TOKENS)


In [ ]:
SUPPORTED_MODELS = set(filter_supported_models(sorted({experiment['model'] for experiment in EXPERIMENTS})))
outfiles_by_group = collections.defaultdict(list)
analysis_nulls = {}


def experiment_outfile(experiment):
    return str(OUTPUT_DIR / f"principle_2_{experiment['name']}.jsonl")


def build_experiment_record(experiment):
    environment_role = experiment.get('environment')
    if environment_role is None:
        environment = None
        role = 'friends'
    else:
        environment, role = environment_role

    model = experiment['model']
    num_common_neighbors = experiment.get('num_common_neighbors', False)
    cot = experiment.get('COT', False)
    er = experiment.get('er', False)
    return {
        'experiment': experiment,
        'name': experiment['name'],
        'model': model,
        'outfile': experiment.get('outfile', experiment_outfile(experiment)),
        'parameters': experiment['parameters'],
        'temperatures': experiment.get('temperatures', DEFAULT_TEMPERATURES),
        'environment': environment,
        'role': role,
        'method': experiment.get('method', 'llm'),
        'num_common_neighbors': num_common_neighbors,
        'cot': cot,
        'cot_config': experiment.get('cot_config'),
        'er': er,
        'summary_group': experiment.get('summary_group', 'sbm'),
        'metadata': {
            'experiment_name': experiment['name'],
            'summary_group': experiment.get('summary_group', 'sbm'),
            'model': model,
            'environment': environment if environment is not None else 'Baseline',
            'role': role,
            'num_common_neighbors': num_common_neighbors,
            'cot': cot,
            'er': er,
        },
    }


experiment_records = []
experiments_to_analyze = []
for experiment in EXPERIMENTS:
    if not experiment.get('run', True):
        continue

    model = experiment['model']
    if model not in SUPPORTED_MODELS:
        print(f'Skipping {experiment["name"]}: {model} is not supported in this environment.')
        continue

    record = build_experiment_record(experiment)
    experiment_records.append(record)

    if experiment.get('include_in_summary', True):
        outfiles_by_group[record['summary_group']].append(record['outfile'])

    if experiment.get('analyze_detail', False):
        experiments_to_analyze.append(record)


if RUN_EXPERIMENTS:
    batch_groups = collections.defaultdict(list)
    for record in experiment_records:
        if record['model'].startswith('Qwen/') and record['method'] == 'llm' and record['experiment'].get('batch', True):
            batch_key = (
                record['model'],
                record['cot'],
                json.dumps(record.get('cot_config'), sort_keys=True, default=str),
            )
            batch_groups[batch_key].append(record)
            continue

        run_network_formation_experiment(
            **record['parameters'],
            outfile=record['outfile'],
            temperatures=record['temperatures'],
            environment=record['environment'],
            role=record['role'],
            method=record['method'],
            num_common_neighbors=record['num_common_neighbors'],
            model=record['model'],
            cot=record['cot'],
            cot_config=record.get('cot_config'),
            er=record['er'],
            metadata=record['metadata'],
        )

    for _, records in batch_groups.items():
        run_network_formation_experiments_batch(records)

if RUN_ANALYSIS:
    for record in experiments_to_analyze:
        group = record['summary_group']
        analysis_nulls[group] = analyze_experiments(
            record['outfile'],
            num_common_neighbors=record['num_common_neighbors'],
            er=record['er'],
            sfx='' if group == 'sbm' else f'_{group}',
        )

    if outfiles_by_group.get('sbm'):
        transitivity_null, probability_null = analysis_nulls.get('sbm', (-1, -1))
        get_table(outfiles_by_group['sbm'], transitivity_null=transitivity_null, probability_null=probability_null)
    if outfiles_by_group.get('er'):
        transitivity_null, probability_null = analysis_nulls.get('er', (-1, -1))
        get_table(outfiles_by_group['er'], transitivity_null=transitivity_null, probability_null=probability_null, er=True, sfx='_er')
    if outfiles_by_group.get('cot'):
        get_table(outfiles_by_group['cot'], sfx='_cot')


All simulations already completed for /content/drive/MyDrive/llm-network-formation-outputs/principle_2_sbm_model_gpt-5-nano_baseline.jsonl. Skipping inference.
Loaded 1 completed simulations from /content/drive/MyDrive/llm-network-formation-outputs/principle_2_sbm_model_Qwen-Qwen3.5-4B_baseline.jsonl
Skipping simulation for n=50, i=0, temperature=default
Loaded 1 completed simulations from /content/drive/MyDrive/llm-network-formation-outputs/principle_2_sbm_model_Qwen-Qwen3.5-4B_school_classmates.jsonl
Skipping simulation for n=50, i=0, temperature=default
Loaded 1 completed simulations from /content/drive/MyDrive/llm-network-formation-outputs/principle_2_sbm_model_Qwen-Qwen3.5-4B_work_colleagues.jsonl
Skipping simulation for n=50, i=0, temperature=default
Loaded 1 completed simulations from /content/drive/MyDrive/llm-network-formation-outputs/principle_2_sbm_model_Qwen-Qwen3.5-4B_community_neighbors.jsonl
Skipping simulation for n=50, i=0, temperature=default
Loaded 1 completed simula

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

INFO 06-28 10:23:37 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 06-28 10:23:37 [nixl_utils.py:34] NIXL is not available
WARNING 06-28 10:23:37 [nixl_utils.py:44] NIXL agent config is not available
INFO 06-28 10:23:37 [model.py:555] Resolved architecture: Qwen3_5ForConditionalGeneration
INFO 06-28 10:23:37 [model.py:1680] Using max model len 262144
INFO 06-28 10:23:37 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-28 10:23:37 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 06-28 10:23:37 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.


video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


WARNING 06-28 10:23:58 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
NEW EDGE {'reason': 'Many mutual friends indicate a strong connection.', 'name': 7}
Adding edge for node 13 in sbm_model_Qwen-Qwen3.5-4B_baseline_cot, simulation=0
NEW EDGE {'reason': 'This candidate has the most mutual friends with me.', 'name': 8}
Adding edge for node 14 in sbm_model_Qwen-Qwen3.5-4B_baseline_cot, simulation=0
NEW EDGE {'reason': 'Strong connection to my existing friends.', 'name': 16}
Adding edge for node 15 in sbm_model_Qwen-Qwen3.5-4B_baseline_cot, simulation=0
NEW EDGE {'reason': 'Many shared friends suggest a strong connection.', 'name': 8}
Adding edge for node 16 in sbm_model_Qwen-Qwen3.5-4B_baseline_cot, simulation=0
NEW EDGE {'reason': 'Many shared friends suggest a good match.'